# LogicNodes Integration for LlamaIndex

This notebook demonstrates how to use **LogicNodes** deterministic compute workers as tools in a **LlamaIndex** agent.

LogicNodes provides 2,300+ cryptographically-signed microservices for autonomous agents:
- Gas oracles (EIP-1559 estimates, deterministic)
- Compliance sentries (on-chain regulatory checks)
- Identity & trust graph scoring
- ZK attestation (proof-of-existence on Base L2)
- DeFi quotes, bridge data, signature verification

**MCP URL:** `https://logicnodes.io/mcp`  
**API Base:** `https://logicnodes.io`  
**Docs:** https://logicnodes.io/docs  
**API key:** https://logicnodes.io/checkout (optional — pay-per-call via USDC x402)

## Install

In [ ]:
%pip install llama-index llama-index-tools-mcp requests -q

## Setup

In [ ]:
import os
import requests
from typing import Any

# Set your API keys
os.environ['OPENAI_API_KEY'] = 'sk-...'  # Replace with your OpenAI key
os.environ['LOGICNODES_API_KEY'] = ''    # Optional: from https://logicnodes.io/checkout

LOGICNODES_API_KEY = os.environ.get('LOGICNODES_API_KEY', '')
LOGICNODES_BASE = 'https://logicnodes.io'
LOGICNODES_MCP_URL = 'https://logicnodes.io/mcp'

def ln_headers() -> dict:
    if LOGICNODES_API_KEY:
        return {'Authorization': f'Bearer {LOGICNODES_API_KEY}'}
    return {}

print('Setup complete.')

## Define LogicNodes Tools

We wrap each LogicNodes worker as a LlamaIndex `FunctionTool`.

In [ ]:
from llama_index.core.tools import FunctionTool


def gas_oracle(chain: str = 'ethereum') -> dict:
    """
    Query the LogicNodes gas oracle for deterministic EIP-1559 gas estimates.
    
    Returns a cryptographically-signed payload with base fee, priority fee,
    and max fee for the specified chain.
    
    Args:
        chain: Chain name. Supported: ethereum, base, polygon, arbitrum.
    """
    resp = requests.post(
        f'{LOGICNODES_BASE}/call/gas-oracle',
        json={'chain': chain},
        headers=ln_headers(),
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()


def compliance_sentry(agent_id: str, action: str, context: str = '') -> dict:
    """
    Run an on-chain compliance check for an autonomous agent action.
    
    Returns a verifiable attestation indicating whether the action is
    permitted under the current regulatory and constitutional ruleset
    anchored by LogicNodes.
    
    Args:
        agent_id: Agent wallet address or DID.
        action: Description of the action to compliance-check.
        context: Optional JSON context for richer analysis.
    """
    resp = requests.post(
        f'{LOGICNODES_BASE}/call/compliance-sentry',
        json={'agent_id': agent_id, 'action': action, 'context': context},
        headers=ln_headers(),
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()


def eth_price() -> dict:
    """
    Fetch the current ETH/USD price from LogicNodes.
    Output is cryptographically signed — suitable for on-chain price verification.
    """
    resp = requests.get(
        f'{LOGICNODES_BASE}/call/eth-price',
        headers=ln_headers(),
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


def zk_attest(content: str) -> dict:
    """
    Anchor content on-chain via LogicNodes ZK attestation.
    
    Returns a verifiable proof-of-existence anchored to Base L2.
    Useful for audit trails, decision logs, and compliance evidence.
    
    Args:
        content: Text or JSON string to anchor on-chain.
    """
    resp = requests.post(
        f'{LOGICNODES_BASE}/x402/zk-attest',
        json={'content': content},
        headers=ln_headers(),
        timeout=20,
    )
    resp.raise_for_status()
    return resp.json()


def graph_score(agent_id: str) -> dict:
    """
    Retrieve the LogicNodes trust graph score for an agent.
    
    Returns a reputation score derived from on-chain interaction history,
    dispute records, and attestation volume.
    
    Args:
        agent_id: Agent wallet address or DID.
    """
    resp = requests.get(
        f'{LOGICNODES_BASE}/graph/score/{agent_id}',
        headers=ln_headers(),
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


# Wrap as LlamaIndex FunctionTools
logicnodes_tools = [
    FunctionTool.from_defaults(fn=gas_oracle, name='gas_oracle'),
    FunctionTool.from_defaults(fn=compliance_sentry, name='compliance_sentry'),
    FunctionTool.from_defaults(fn=eth_price, name='eth_price'),
    FunctionTool.from_defaults(fn=zk_attest, name='zk_attest'),
    FunctionTool.from_defaults(fn=graph_score, name='graph_score'),
]

print(f'Registered {len(logicnodes_tools)} LogicNodes tools.')

## Build the LlamaIndex Agent

In [ ]:
from llama_index.core.agent import ReActAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)

agent = ReActAgent.from_tools(
    logicnodes_tools,
    llm=llm,
    verbose=True,
    system_prompt=(
        'You are an autonomous on-chain agent powered by LogicNodes deterministic '
        'compute. Before recommending any on-chain action:\n'
        '1. Call compliance_sentry to verify the action is permitted.\n'
        '2. Call gas_oracle to estimate transaction costs.\n'
        '3. Call eth_price for current ETH valuation.\n'
        '4. Anchor critical decisions with zk_attest for audit trails.\n'
        '5. Check graph_score to assess counterparty reputation.\n\n'
        'All LogicNodes responses are cryptographically signed and verifiable on Base L2.'
    ),
)

print('Agent built successfully.')

## Run the Agent

In [ ]:
response = agent.chat(
    'I want to swap 500 USDC for ETH on Base. '
    'Check: (1) current ETH price, '
    '(2) gas estimate on Base, '
    '(3) compliance for agent "llama-demo" performing "swap 500 USDC to ETH on Base", '
    '(4) graph score for counterparty 0xUniswapRouterBase. '
    'Should I proceed and at what parameters?'
)

print('\n=== Agent Response ===')
print(str(response))

## Option B: Dynamic Tool Discovery via MCP

Connect directly to the LogicNodes MCP server to dynamically discover and use all 2,300+ workers.

In [ ]:
# Requires: pip install llama-index-tools-mcp
# from llama_index.tools.mcp import MCPToolSpec

# mcp_tool_spec = MCPToolSpec(url=LOGICNODES_MCP_URL)
# mcp_tools = mcp_tool_spec.to_tool_list()
# print(f'Discovered {len(mcp_tools)} tools via MCP')

# mcp_agent = ReActAgent.from_tools(mcp_tools, llm=llm, verbose=True)

print('MCP integration: uncomment the lines above after installing llama-index-tools-mcp')

## Resources

- **LogicNodes API docs:** https://logicnodes.io/docs
- **LogicNodes MCP server:** `npx -y @logicnodes/mcp-server`
- **Worker catalog:** https://logicnodes.io/workers
- **API key / pricing:** https://logicnodes.io/checkout
- **LlamaIndex docs:** https://docs.llamaindex.ai